In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        ChatMessage(role=ChatMessageRole.USER, content="Reply with exactly: LLM is working")
    ]
)
print(response.choices[0].message.content)

LLM is working


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import re

w = WorkspaceClient()
LLM = "databricks-meta-llama-3-3-70b-instruct"   # the model that just worked

# --- schema context: tell the LLM what it can query ---
SCHEMA_CONTEXT = """
You have access to these data objects. Generate ONE read-only Spark SQL SELECT query to answer the question.

1. TABLE `silver_financials` (structured, transactional detail):
   columns: business_unit, business_unit_name, cost_centre, cost_centre_name,
   month (DATE), fiscal_year, fiscal_month, gl_account, account_type,
   actual_amount_aud, transaction_count

2. METRIC VIEW `finance_metrics` (governed semantic layer - USE THIS for KPIs/totals):
      dimensions (use these EXACT names):
     - business_unit  (values: 'Northern Region','Southern Region','Eastern Region','Western Region')
     - month  (DATE, first of month e.g. '2026-05-01')
     - fiscal_year  (e.g. 'FY2026')
     - scenario  (values: 'Budget','Actual','Forecast')
     - kpi  (values: 'Total Revenue','Total Cost','Gross Margin','EBITDA')
   IMPORTANT: the dimension is 'business_unit' (NOT 'business_unit_name'). Filter like: WHERE business_unit = 'Northern Region'
   RULES for the metric view:
   - Measures MUST be wrapped in MEASURE(): e.g. MEASURE(total_value)
   - GROUP BY only dimensions, never measures
   - 'scenario' values are 'Budget','Actual','Forecast'
   - 'kpi' values include 'Total Revenue','Total Cost','Gross Margin','EBITDA'

   CRITICAL - comparing scenarios (Budget vs Actual):
NEVER alias the same measure into two columns (that always gives 0).
To compare scenarios, GROUP BY scenario so each is a separate ROW.

CORRECT pattern for a "revenue variance vs budget" question:
  SELECT scenario, MEASURE(total_value) AS revenue
  FROM finance_metrics
  WHERE business_unit = 'Northern Region'
    AND kpi = 'Total Revenue'
    AND month = '2026-05-01'
  GROUP BY scenario

This returns separate Budget and Actual rows. The variance is the Actual row minus the Budget row - explain that in the answer, do not compute it in SQL by subtracting a measure from itself.

Guidance:
- For plan-vs-actual, variance, or KPI questions -> use the metric view finance_metrics.
- For transaction-level or cost-centre detail -> use silver_financials.
- Business unit names include 'Northern Region','Southern Region','Eastern Region','Western Region'.
- Dates are first-of-month, e.g. '2026-05-01'.

Return ONLY the SQL query, no explanation, no markdown fences.
"""

def generate_sql(question):
    resp = w.serving_endpoints.query(
        name=LLM,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=SCHEMA_CONTEXT),
            ChatMessage(role=ChatMessageRole.USER, content=question)
        ]
    )
    sql = resp.choices[0].message.content.strip()
    # strip any markdown fences the model might add
    sql = re.sub(r"^```sql\s*|\s*```$", "", sql).strip()
    return sql

def is_read_only(sql):
    lowered = sql.lower()
    banned = ["insert","update","delete","drop","alter","create","truncate","merge","grant"]
    return sql.lower().strip().startswith("select") and not any(b in lowered for b in banned)

def ask_data(question):
    sql = generate_sql(question)
    print("Generated SQL:\n", sql, "\n")
    if not is_read_only(sql):
        return "Blocked: only read-only SELECT queries are allowed."
    try:
        result = spark.sql(sql)
        rows = result.limit(50).toPandas().to_string(index=False)
    except Exception as e:
        return f"Query error: {e}"
    # explain the result
    explain = w.serving_endpoints.query(
        name=LLM,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content="You are a finance analyst. Explain the query result clearly and concisely, with the key numbers."),
            ChatMessage(role=ChatMessageRole.USER, content=f"Question: {question}\n\nData:\n{rows}")
        ]
    )
    return explain.choices[0].message.content

In [0]:
print(ask_data("What was Northern Region's revenue variance versus budget in May?"))

Generated SQL:
 SELECT scenario, MEASURE(total_value) AS revenue 
FROM finance_metrics 
WHERE business_unit = 'Northern Region' 
  AND kpi = 'Total Revenue' 
  AND month = '2026-05-01' 
GROUP BY scenario 

The revenue variance for the Northern Region in May is a shortfall of $210,470. 

This is calculated by subtracting the Actual revenue ($404,530) from the Budget revenue ($615,000). The variance as a percentage is approximately 34.2% ($210,470 / $615,000).


In [0]:
print(ask_data("Which cost centres in Northern Region had the highest actual spend?"))

Generated SQL:
 SELECT cost_centre, cost_centre_name, SUM(actual_amount_aud) AS total_spend 
FROM silver_financials 
WHERE business_unit = 'Northern Region' 
GROUP BY cost_centre, cost_centre_name 
ORDER BY total_spend DESC 

Based on the data, the cost centre with the highest actual spend in the Northern Region is:

**Sales - North (CC-1010)** with a total spend of **£2,619,049.21**.

This is approximately 1.8 times higher than the spend of the next highest cost centre, **Admin - North (CC-1030)**, which had a total spend of **£1,454,699.86**.


In [0]:
print(ask_data("Compare gross margin across all business units for May, budget versus actual."))

Generated SQL:
 SELECT business_unit, scenario, MEASURE(total_value) AS gross_margin
FROM finance_metrics
WHERE month = '2026-05-01'
  AND fiscal_year = 'FY2026'
  AND kpi = 'Gross Margin'
GROUP BY business_unit, scenario 

**Gross Margin Comparison Across Business Units (May)**

Here are the key numbers:

* **Northern Region**: Actual gross margin ($89,732) is 36% below Budget ($139,400)
* **Southern Region**: Actual gross margin ($178,565) is 2% above Budget ($175,100)
* **Eastern Region**: Actual gross margin ($165,136) is 1% below Budget ($166,600)
* **Western Region**: Actual gross margin ($175,402) is 2% below Budget ($178,500)

**Summary**:

* Southern Region is the only unit to exceed its budgeted gross margin.
* Northern Region has the largest shortfall, with actual gross margin 36% below budget.
* Eastern and Western Regions are slightly below budget, with shortfalls of 1% and 2%, respectively.


In [0]:
%sql
SELECT scenario,
       MEASURE(total_value) AS revenue,
       MEASURE(total_variance) AS variance
FROM finance_metrics
WHERE business_unit = 'Northern Region'
  AND kpi = 'Total Revenue'
  AND month = '2026-05-01'
GROUP BY scenario
ORDER BY scenario;

scenario,revenue,variance
Actual,404530,-210470
Budget,615000,null
Forecast,606680,null


In [0]:
%sql
CREATE OR REPLACE VIEW finance_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Governed financial KPI semantic layer. Query these measures for plan-vs-actual, variance, and KPI analysis."
source: gold_kpi
dimensions:
  - name: business_unit
    expr: business_unit_name
    comment: "Business unit full name, e.g. 'Northern Region'. Use to filter or group by unit."
  - name: month
    expr: month
    comment: "Reporting month as first-of-month DATE, e.g. '2026-05-01'."
  - name: fiscal_year
    expr: fiscal_year
    comment: "Australian fiscal year label, e.g. 'FY2026' (FY runs Jul-Jun)."
  - name: scenario
    expr: scenario
    comment: "Plan scenario: 'Budget', 'Actual', or 'Forecast'."
  - name: kpi
    expr: kpi_name
    comment: "KPI name: 'Total Revenue', 'Total Cost', 'Gross Margin', or 'EBITDA'."
measures:
  - name: total_value
    expr: SUM(value_aud)
    comment: "KPI value in AUD. Wrap in MEASURE(). Group by scenario to compare Budget vs Actual."
  - name: total_variance
    expr: SUM(variance_vs_budget)
    comment: "Variance vs budget in AUD (Actual minus Budget). Only meaningful on Actual-scenario rows."
$$;

In [0]:
display(spark.sql("DESCRIBE TABLE finance_metrics"))

col_name,data_type,comment,metadata
business_unit,string,"Business unit full name, e.g. 'Northern Region'. Use to filter or group by unit.",null
month,date,"Reporting month as first-of-month DATE, e.g. '2026-05-01'.",null
fiscal_year,string,"Australian fiscal year label, e.g. 'FY2026' (FY runs Jul-Jun).",null
scenario,string,"Plan scenario: 'Budget', 'Actual', or 'Forecast'.",null
kpi,string,"KPI name: 'Total Revenue', 'Total Cost', 'Gross Margin', or 'EBITDA'.",null
total_value,bigint measure,KPI value in AUD. Wrap in MEASURE(). Group by scenario to compare Budget vs Actual.,null
total_variance,bigint measure,Variance vs budget in AUD (Actual minus Budget). Only meaningful on Actual-scenario rows.,null


In [0]:
def get_table_metadata(table_name):
    """Read column names, types, and comments from the catalog for one table/view."""
    cols = spark.sql(f"DESCRIBE TABLE {table_name}").collect()
    lines = []
    for row in cols:
        col, dtype, comment = row['col_name'], row['data_type'], row['comment']
        # DESCRIBE includes blank/partition rows - skip them
        if not col or col.startswith('#') or col.strip() == '':
            continue
        desc = f" -- {comment}" if comment else ""
        lines.append(f"    {col} ({dtype}){desc}")
    return "\n".join(lines)

def get_table_comment(table_name):
    """Read the table-level description."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {table_name}").collect()
    for r in rows:
        if r['col_name'] == 'Comment':
            return r['data_type']  # the value sits in data_type column for this row
    return ""

def build_schema_context():
    """Assemble the full schema context from catalog metadata - no hardcoding."""
    sources = {
        "finance_metrics": "METRIC VIEW (governed semantic layer - PREFER for KPIs, variance, plan-vs-actual)",
        "silver_financials": "TABLE (transactional detail - use for cost-centre/account drill-down)",
    }
    parts = ["You answer finance questions by generating ONE read-only Spark SQL SELECT query.\n"]
    parts.append("Available data sources:\n")
    for tbl, role in sources.items():
        tcomment = get_table_comment(tbl)
        parts.append(f"\n`{tbl}` - {role}")
        if tcomment:
            parts.append(f"  Description: {tcomment}")
        parts.append("  Columns:")
        parts.append(get_table_metadata(tbl))
    # the non-negotiable rules that aren't in column comments
    parts.append("""
RULES:
- For the metric view `finance_metrics`: wrap measures in MEASURE(), e.g. MEASURE(total_value). GROUP BY dimensions only.
- To compare Budget vs Actual: GROUP BY scenario so they are separate ROWS. Never alias the same measure into two columns.
- All amounts are AUD. Never use other currency symbols.
- Return ONLY the SQL, no markdown, no explanation.
""")
    parts.append(FEW_SHOT_EXAMPLES)
    return "\n".join(parts)


In [0]:

# build it and inspect
SCHEMA_CONTEXT = build_schema_context()
print(SCHEMA_CONTEXT)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8658595524745892>, line 3
      2 # build it and inspect
----> 3 SCHEMA_CONTEXT = build_schema_context()
      4 print(SCHEMA_CONTEXT)

File <command-8658595524745891>, line 45, in build_schema_context()
     37     # the non-negotiable rules that aren't in column comments
     38     parts.append("""
     39 RULES:
     40 - For the metric view `finance_metrics`: wrap measures in MEASURE(), e.g. MEASURE(total_value). GROUP BY dimensions only.
   (...)     43 - Return ONLY the SQL, no markdown, no explanation.
     44 """)
---> 45     parts.append(FEW_SHOT_EXAMPLES)
     46     return "\n".join(parts)

NameError: name 'FEW_SHOT_EXAMPLES' is not defined

In [0]:
FEW_SHOT_EXAMPLES = """
Here are examples of correct queries for common question types:

EXAMPLE 1 - Variance/budget-vs-actual (metric view, group by scenario as ROWS):
Q: What was Northern Region's revenue variance versus budget in May?
SQL: SELECT scenario, MEASURE(total_value) AS revenue
     FROM finance_metrics
     WHERE business_unit = 'Northern Region' AND kpi = 'Total Revenue' AND month = '2026-05-01'
     GROUP BY scenario

EXAMPLE 2 - Compare a KPI across all units (metric view, group by unit AND scenario):
Q: Compare gross margin across all business units for May, budget versus actual.
SQL: SELECT business_unit, scenario, MEASURE(total_value) AS gross_margin
     FROM finance_metrics
     WHERE kpi = 'Gross Margin' AND month = '2026-05-01'
     GROUP BY business_unit, scenario
     ORDER BY business_unit

EXAMPLE 3 - Transaction/cost-centre detail (silver table, not the metric view):
Q: Which cost centres in Northern Region had the highest actual spend?
SQL: SELECT cost_centre, cost_centre_name, SUM(actual_amount_aud) AS total_spend
     FROM silver_financials
     WHERE business_unit = 'Northern Region'
     GROUP BY cost_centre, cost_centre_name
     ORDER BY total_spend DESC

EXAMPLE 4 - Ranking across units (metric view):
Q: Which business unit had the worst revenue variance in May?
SQL: SELECT business_unit, scenario, MEASURE(total_value) AS revenue
     FROM finance_metrics
     WHERE kpi = 'Total Revenue' AND month = '2026-05-01'
     GROUP BY business_unit, scenario
     ORDER BY business_unit
"""

In [0]:
print(ask_data("Compare total cost between Northern and Southern Region in May, actual only."))

Generated SQL:
 SELECT business_unit, MEASURE(total_value) AS total_cost
FROM finance_metrics
WHERE scenario = 'Actual'
  AND kpi = 'Total Cost'
  AND month = '2026-05-01'
  AND business_unit IN ('Northern Region', 'Southern Region')
GROUP BY business_unit 

Based on the actual data for May, the total cost comparison between the Northern and Southern Regions is as follows:

* Northern Region: $467,452
* Southern Region: $582,710

The Southern Region has a higher total cost, exceeding the Northern Region by $115,258.


In [0]:
EVAL_SET = [
    {
        "question": "What was Northern Region's revenue variance versus budget in May?",
        "expect_contains": ["210", "404", "615"],   # key numbers that must appear
        "expect_source": "finance_metrics",
    },
    {
        "question": "Which cost centres in Northern Region had the highest actual spend?",
        "expect_contains": ["Sales - North"],
        "expect_source": "silver_financials",
    },
    {
        "question": "Compare gross margin across all business units for May, budget versus actual.",
        "expect_contains": ["Northern", "Southern", "Eastern", "Western"],
        "expect_source": "finance_metrics",
    },
    {
        "question": "Which business unit had the worst revenue variance in May?",
        "expect_contains": ["Northern"],
        "expect_source": "finance_metrics",
    },
    {
        "question": "Compare total cost between Northern and Southern Region in May, actual only.",
        "expect_contains": ["Northern", "Southern"],
        "expect_source": "finance_metrics",
    },
    {
        "question": "What is the total actual revenue across all business units in May?",
        "expect_contains": [],   
        "expect_source": "finance_metrics",
    },
]

In [0]:
def run_evaluation():
    results = []
    for i, case in enumerate(EVAL_SET, 1):
        q = case["question"]
        sql = generate_sql(q)
        source_ok = case["expect_source"] in sql.lower()
        # run the FULL agent (SQL + explanation) and check the final answer text
        answer = ""
        error = None
        try:
            if is_read_only(sql):
                answer = ask_data(q)   # full answer incl. explanation
            else:
                error = "blocked"
        except Exception as e:
            error = str(e)[:100]
        # strip commas from numbers so "210,470" matches "210"
        answer_norm = answer.replace(",", "")
        content_ok = all(str(k) in answer_norm for k in case["expect_contains"])
        passed = source_ok and content_ok and error is None
        results.append({"#": i, "question": q[:40],
                        "source_ok": source_ok, "content_ok": content_ok,
                        "passed": passed, "error": error})
    import pandas as pd
    df = pd.DataFrame(results)
    p = df["passed"].sum()
    print(f"\n{'='*55}\nEVALUATION: {p}/{len(EVAL_SET)} passed ({100*p//len(EVAL_SET)}%)\n{'='*55}")
    return df

In [0]:
display(run_evaluation())

Generated SQL:
 SELECT scenario, MEASURE(total_value) AS revenue 
FROM finance_metrics 
WHERE business_unit = 'Northern Region' 
  AND kpi = 'Total Revenue' 
  AND month = '2026-05-01' 
GROUP BY scenario 

Generated SQL:
 SELECT cost_centre, cost_centre_name, SUM(actual_amount_aud) AS total_spend 
FROM silver_financials 
WHERE business_unit = 'Northern Region' 
GROUP BY cost_centre, cost_centre_name 
ORDER BY total_spend DESC 

Generated SQL:
 SELECT business_unit, scenario, MEASURE(total_value) AS gross_margin
FROM finance_metrics
WHERE month = '2026-05-01'
  AND kpi = 'Gross Margin'
  AND scenario IN ('Budget', 'Actual')
GROUP BY business_unit, scenario 

Generated SQL:
 SELECT business_unit, scenario, MEASURE(total_value) AS revenue
FROM finance_metrics
WHERE kpi = 'Total Revenue'
  AND month = '2026-05-01'
GROUP BY business_unit, scenario
ORDER BY business_unit, scenario 

Generated SQL:
 SELECT business_unit, MEASURE(total_value) AS total_cost
FROM finance_metrics
WHERE kpi = 'Tot

#,question,source_ok,content_ok,passed,error
1,What was Northern Region's revenue varia,true,true,true,null
2,Which cost centres in Northern Region ha,true,true,true,null
3,Compare gross margin across all business,true,true,true,null
4,Which business unit had the worst revenu,true,true,true,null
5,Compare total cost between Northern and,true,true,true,null
6,What is the total actual revenue across,true,true,true,null


In [0]:
def search_documents(question, k=3):
    """Semantic search over email/SharePoint commentary for the 'why' behind the numbers."""
    safe_q = question.replace("'", "''")
    results = spark.sql(f"""
        SELECT title, source, content FROM vector_search(
          index => 'workspace.default.document_search',
          query_text => '{safe_q}',
          num_results => {k}
        )
    """).toPandas()
    if results.empty:
        return "No relevant documents found."
    return "\n\n".join(f"[{r['source']} - {r['title']}]\n{r['content']}" for _, r in results.iterrows())

# quick test
print(search_documents("Why did Northern Region miss its revenue target?"))

[sharepoint - Budget commentary - Northern Region May]
Business Unit: Northern Region. Cost Centre: CC-1010. Month: 2026-05. Variance Flagged: Revenue -35% vs budget. Commentary: Major client Customer-482 delayed contract renewal from May to July; two large deals slipped out of quarter. Pipeline intact but timing hit May revenue hard. Owner: J. Fletcher. Action: Renewal rescheduled July; forecast recovery Q3.

[email - Northern Region May revenue shortfall explanation]
From James Fletcher, 4 June 2026. Northern Region's May revenue came in about 35% below budget for cost centre CC-1010. Root cause: our largest Northern account, Customer-482, pushed their contract renewal from May into July - a timing decision tied to their budget cycle, not a loss. Two mid-size deals forecast to close in May also slipped to early Q3. Impact: about $310k of expected May revenue moved out of the quarter. Cost lines were on plan. Outlook: pipeline intact, Customer-482 renewal now contracted for July, reco

In [0]:
def finance_agent(question):
    # Step 1: router - decide which tools are needed
    routing_resp = w.serving_endpoints.query(
        name=LLM,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content="""Decide which data tools are needed to answer a finance question.
Reply with ONLY a comma-separated list from: SQL, DOCS
- SQL: for numbers, KPIs, variances, amounts, comparisons (structured data)
- DOCS: for reasons, explanations, commentary, 'why' questions (documents/email)
Many questions need BOTH.
Examples:
'what was the revenue variance' -> SQL
'why did revenue drop' -> DOCS
'what was the variance and why' -> SQL, DOCS"""),
            ChatMessage(role=ChatMessageRole.USER, content=question)
        ]
    )
    routing = routing_resp.choices[0].message.content.upper()

    context_parts = []

    # Step 2: call the chosen tools
    if "SQL" in routing:
        sql = generate_sql(question)
        if is_read_only(sql):
            try:
                rows = spark.sql(sql).limit(50).toPandas().to_string(index=False)
                context_parts.append(f"STRUCTURED DATA (from SQL):\n{rows}")
            except Exception as e:
                context_parts.append(f"SQL error: {e}")
    if "DOCS" in routing:
        context_parts.append(f"DOCUMENT CONTEXT (email/SharePoint):\n{search_documents(question)}")

    # Step 3: synthesize final answer
    combined = "\n\n".join(context_parts)
    final_resp = w.serving_endpoints.query(
        name=LLM,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content="You are a finance analyst. Answer the question using the data provided. Combine the numbers and the commentary into one clear answer. All amounts are in AUD."),
            ChatMessage(role=ChatMessageRole.USER, content=f"Question: {question}\n\n{combined}")
        ]
    )
    print(f"[Router chose: {routing}]\n")
    return final_resp.choices[0].message.content

In [0]:
print(finance_agent("Northern Region's revenue was down in May. What was the variance, and why did it happen?"))

[Router chose: SQL, DOCS]

The variance in Northern Region's revenue for May was approximately $210,470, which is 34.2% below budget ($615,000 - $404,530). This occurred because the region's largest account, Customer-482, delayed their contract renewal from May to July, resulting in about $310,000 of expected revenue being pushed out of the quarter. Additionally, two mid-size deals that were forecast to close in May slipped to early Q3, further contributing to the revenue shortfall. However, this is considered a timing variance rather than a demand problem, and recovery is expected in Q3.


In [0]:
print(finance_agent("Compare total cost between Northern and Southern Region in May, actual only"))

[Router chose: SQL]

The total cost for the Northern Region in May was $467,452, while the total cost for the Southern Region was $582,710. This means the Southern Region had a higher total cost, exceeding the Northern Region by $115,258.
